# 11 · Draft composites: updrafts, downdrafts and the rest, vs r_v

At every 3D snapshot, each column is sorted into groups by a **mask** (default: W at the second level, |W| > X). Every field, plus MSE, frozen MSE, SAM's liquid/ice static energy and θₑ, is then averaged over each group, level by level. This is done for every r_v, so we can see how each group changes as the surface dries.

1. Run **setup** and **CONFIG**.
2. Pick or write a mask in the **MASK BOX**. Tune it on the cheap **preview**.
3. **Compute**. The first time takes a few minutes per run; results are cached per mask definition.
4. Plots: **A** one run, every quantity, by group; **B** each group across r_v (optionally as anomalies from the domain mean); **C** summary vs RH; **D** local stability (Emanuel 6.1.7 / 6.2.10).

The engine is in `samheat/analysis/composites.py` (tested in `tests/test_composites.py`). Your original testing cells are in the appendix at the bottom.

In [ ]:
# RUN THIS CELL FIRST: lets this notebook find the samheat code in this repo (no install needed)
import os, sys, pathlib
def _find_repo():
    starts = [pathlib.Path(os.environ.get("JPY_SESSION_NAME", ".")).resolve(), pathlib.Path.cwd().resolve()]
    for s in starts:
        for d in [s, *s.parents]:
            if (d / "samheat" / "__init__.py").exists():
                return d
    for d in [pathlib.Path("/scratch/ttg2015/Torch/ExtremeHeat/SAM_HEAT"), pathlib.Path.home() / "SAM_HEAT"]:
        if (d / "samheat" / "__init__.py").exists():
            return d
    raise FileNotFoundError("Could not find the SAM_HEAT folder. Set REPO by hand, e.g. REPO = pathlib.Path('/scratch/.../SAM_HEAT')")
REPO = _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import samheat
print("using samheat from", pathlib.Path(samheat.__file__).parent)

In [ ]:
# ---- CONFIG ----
from samheat.paths import SITE
EXPERIMENT = SITE.work_root / "vdd2025_vddgrid"     # stock-grid runs: "vdd2025_repro"
RV_LIST = [0, 200, 500, 1000, 2000]                 # one stage2_rv<rv> run each
SOURCE = "auto"           # "auto": converted .nc files when every snapshot has one, else .bin3D;
                          # "nc" / "bin" to force one (identical data, so the cache is shared)
TMIN, TMAX = 30, 60       # days of Stage 2 to use (3-hourly 3D snapshots)
INCLUDE_ALL = True        # also composite the whole domain as group "all" (for anomalies)
ZMAX_PLOT = 16000         # m, top of the profile plots (None = model top)
RV_COLORS = ["#1a5292", "#5fa5dd", "#446400", "#d9863b", "#783c25"]   # same as 02_Repro

## Mask box
Edit or add mask definitions here. Switch with `MASK` and `MASK_PARAMS`.

In [ ]:
# ===================== MASK BOX =====================
# A mask function gets ONE snapshot and returns {group name: boolean (y, x) array}.
#   f : dict of 3D arrays (z, y, x), float64, SAM units: U V W PP QRAD TABS QV QN QP
#       plus derived MSE MSE_frozen S_LI THETA_E          z : (z,) heights in m
# Write your own below; groups must be the same every snapshot (they may overlap).
# Changing a function's code or its parameters automatically gives a new cache entry.
import numpy as np

def symmetric(f, z, k=1, X=0.3):
    """|W| > X at level k (your first definition)."""
    w = f["W"][k]
    return dict(updraft=w > X, downdraft=w < -X, intermediate=np.abs(w) <= X)

def sigma(f, z, k=1, n=2.0):
    """|W| > n standard deviations of W at level k in this snapshot (adapts to each run)."""
    w = f["W"][k]
    X = n * w.std()
    return dict(updraft=w > X, downdraft=w < -X, intermediate=np.abs(w) <= X)

def percentile(f, z, k=1, q=1.0):
    """Strongest q % of upward and of downward W at level k in this snapshot."""
    w = f["W"][k]
    hi, lo = np.percentile(w, [100 - q, q])
    return dict(updraft=w >= hi, downdraft=w <= lo, intermediate=(w > lo) & (w < hi))

def layer_mean(f, z, zbot=0., ztop=1000., X=0.3):
    """|W| > X for W averaged over the levels between zbot and ztop (m)."""
    sel = (z >= zbot) & (z <= ztop)
    w = f["W"][sel].mean(axis=0)
    return dict(updraft=w > X, downdraft=w < -X, intermediate=np.abs(w) <= X)

def cloudy(f, z, k=1, X=0.3, k_cloud=10, qn_min=0.01):
    """Updrafts at level k split by whether the column is cloudy at level k_cloud (QN > qn_min g/kg)."""
    w, cloud = f["W"][k], f["QN"][k_cloud] > qn_min
    return dict(cloudy_updraft=(w > X) & cloud, clear_updraft=(w > X) & ~cloud,
                downdraft=w < -X, intermediate=np.abs(w) <= X)

MASKS = dict(symmetric=symmetric, sigma=sigma, percentile=percentile, layer_mean=layer_mean, cloudy=cloudy)

# ---- choose one ----
MASK = "symmetric"
MASK_PARAMS = dict(k=1, X=0.3)
print(f"mask: {MASK}{MASK_PARAMS}")

## Preview the mask (one run, a few snapshots)

In [ ]:
# ---- Preview the mask on a few snapshots of one run (cheap: tune the definition here first) ----
import numpy as np, matplotlib.pyplot as plt
from samheat.analysis.composites import open_3d, derived_fields
PREVIEW_RV, N_SNAP = RV_LIST[0], 8

d = open_3d(EXPERIMENT / f"stage2_rv{PREVIEW_RV:g}", SOURCE, TMIN, TMAX)
z = d.z.values.astype(float)
k = MASK_PARAMS.get("k", 1)
idx = np.linspace(0, d.sizes["time"] - 1, N_SNAP).astype(int)
counts, wk = {}, []
for t in idx:
    s = d.isel(time=t).load()
    f = {v: s[v].values.astype(float) for v in d.data_vars if d[v].dims == ("time", "z", "y", "x")}
    f |= derived_fields(f, np.asarray(s["p"].values, float).reshape(-1, len(z))[0], z)
    m = MASKS[MASK](f, z, **MASK_PARAMS)
    for g, a in m.items():
        counts[g] = counts.get(g, 0) + a.sum()
    wk.append(f["W"][k].ravel())
total = len(idx) * f["W"][k].size
print(f"rv={PREVIEW_RV}: {N_SNAP} snapshots, W at z index {k} ({z[k]:.0f} m label)")
for g, c in counts.items():
    print(f"  {g:16s} {100 * c / total:6.2f} %")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
im = ax[0].pcolormesh(d.x / 1e3, d.y / 1e3, f["W"][k], cmap="RdBu_r", vmin=-1, vmax=1)
fig.colorbar(im, ax=ax[0], label="W (m/s)"); ax[0].set_title(f"W at z index {k}, day {float(s.time):.2f}")
cat = np.full(f["W"][k].shape, np.nan)
for i, (g, a) in enumerate(m.items()):
    cat[a] = i
im = ax[1].pcolormesh(d.x / 1e3, d.y / 1e3, cat, cmap=plt.get_cmap("tab10", len(m)), vmin=-.5, vmax=len(m) - .5)
cb = fig.colorbar(im, ax=ax[1], ticks=range(len(m))); cb.ax.set_yticklabels(list(m))
ax[1].set_title("groups (same snapshot; overlapping groups: last wins)")
ax[2].hist(np.concatenate(wk), bins=400, log=True); ax[2].set_xlabel(f"W at z index {k} (m/s)")
ax[2].set_title(f"{N_SNAP} snapshots")
for a in ax[:2]:
    a.set_xlabel("x (km)"); a.set_ylabel("y (km)"); a.set_aspect("equal")
plt.tight_layout(); plt.show()

## Compute composites for every r_v

In [ ]:
# ---- Composites for every r_v (cached per run in <experiment>/analysis_cache; first time: minutes per run) ----
from samheat.analysis.composites import composite_experiment
C = composite_experiment(EXPERIMENT, RV_LIST, MASKS[MASK], MASK, MASK_PARAMS,
                         source=SOURCE, tmin=TMIN, tmax=TMAX, include_all=INCLUDE_ALL)
C

In [ ]:
# ---- Group fractions (% of columns) vs r_v and near-surface RH ----
import pandas as pd
frac = (100 * C.fraction).to_pandas().round(2)
frac.insert(0, "RH (%)", C.RH.values.round(1))
frac

## A. One run, all quantities by group

In [ ]:
# ---- A. One run: every quantity, one line per group ----
# x-axes fit the data between 0 and ZMAX_PLOT; dashed = saturated counterpart (MSE*, theta_e*)
import matplotlib.pyplot as plt
from samheat.analysis.composites import SAT_PAIRS, fit_xlim
RV_SHOW = RV_LIST[0]

c = C.sel(rv=RV_SHOW)
qs = [q for q in C.data_vars if q not in SAT_PAIRS.values()]
ncol = 4
nrow = -(-len(qs) // ncol)
fig, axs = plt.subplots(nrow, ncol, figsize=(4 * ncol, 4.2 * nrow), sharey=True, squeeze=False)
for ax, q in zip(axs.flat, qs):
    shown = []
    for i, g in enumerate(c.group.values):
        prof = c[q].sel(group=g)
        ax.plot(prof, c.z / 1000, c=f"C{i}", label=f"{g} ({100 * float(c.fraction.sel(group=g)):.1f} %)")
        shown.append(prof.values)
        if q in SAT_PAIRS and SAT_PAIRS[q] in c:
            sat = c[SAT_PAIRS[q]].sel(group=g)
            ax.plot(sat, c.z / 1000, c=f"C{i}", ls="--", lw=1)
            shown.append(sat.values)
    fit_xlim(ax, shown, c.z, ZMAX_PLOT)
    ax.set_title(q + (" (dashed: saturated)" if q in SAT_PAIRS else ""))
for ax in axs[:, 0]:
    ax.set_ylabel("z (km)")
for ax in axs.flat[len(qs):]:
    ax.set_visible(False)
if ZMAX_PLOT:
    axs[0, 0].set_ylim(0, ZMAX_PLOT / 1000)
axs.flat[0].legend(fontsize=8)
fig.suptitle(f"rv = {RV_SHOW:g} s/m (RH {float(c.RH):.1f} %), mask {MASK}{MASK_PARAMS}")
fig.tight_layout(); plt.show()

## B. Each group across r_v

In [ ]:
# ---- B. How each group changes with r_v: one figure per group, one line per r_v ----
# x-axes fit the data between 0 and ZMAX_PLOT; dashed = saturated counterpart (MSE*, theta_e*)
import matplotlib.pyplot as plt
from samheat.analysis.composites import SAT_PAIRS, fit_xlim
GROUPS_SHOW = [g for g in C.group.values if g != "all"]
QUANTS = ["W", "TABS", "QV", "QN", "QP", "MSE", "MSE_frozen", "THETA_E"]
ANOMALY = False           # True: plot group minus the domain mean ("all") of the same run
SHOW_SAT = True           # dashed saturated lines on the MSE / theta_e panels

for g in GROUPS_SHOW:
    ncol = 4
    nrow = -(-len(QUANTS) // ncol)
    fig, axs = plt.subplots(nrow, ncol, figsize=(4 * ncol, 4.2 * nrow), sharey=True, squeeze=False)
    for ax, q in zip(axs.flat, QUANTS):
        shown = []
        names = [q] + ([SAT_PAIRS[q]] if SHOW_SAT and q in SAT_PAIRS and SAT_PAIRS[q] in C else [])
        for i, rv in enumerate(C.rv.values):
            for j, name in enumerate(names):
                prof = C[name].sel(rv=rv, group=g)
                if ANOMALY:
                    prof = prof - C[name].sel(rv=rv, group="all")
                ax.plot(prof, C.z / 1000, c=RV_COLORS[i % len(RV_COLORS)], ls="--" if j else "-", lw=1 if j else 1.5,
                        label=None if j else f"rv {rv:g} (RH {float(C.RH.sel(rv=rv)):.0f} %, {100 * float(C.fraction.sel(rv=rv, group=g)):.1f} %)")
                shown.append(prof.values)
        fit_xlim(ax, shown, C.z, ZMAX_PLOT)
        ax.set_title(q + (" - all" if ANOMALY else "") + (" (dashed: saturated)" if len(names) > 1 else ""))
        if ANOMALY:
            ax.axvline(0, c=".7", lw=.5)
    for ax in axs[:, 0]:
        ax.set_ylabel("z (km)")
    for ax in axs.flat[len(QUANTS):]:
        ax.set_visible(False)
    if ZMAX_PLOT:
        axs[0, 0].set_ylim(0, ZMAX_PLOT / 1000)
    axs.flat[0].legend(fontsize=7)
    fig.suptitle(f"{g}: mask {MASK}{MASK_PARAMS}")
    fig.tight_layout(); plt.show()

## C. Summary vs near-surface RH

In [ ]:
# ---- C. Summary vs near-surface RH: group fraction, and W of each group at the mask level ----
import matplotlib.pyplot as plt
k = MASK_PARAMS.get("k", 1)
groups = [g for g in C.group.values if g != "all"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for g in groups:
    ax[0].plot(C.RH, 100 * C.fraction.sel(group=g), "o-", label=g)
    ax[1].plot(C.RH, C.W.sel(group=g).isel(z=k), "o-", label=g)
ax[0].set_ylabel("% of columns"); ax[1].set_ylabel(f"mean W at z index {k} (m/s)")
for a in ax:
    a.set_xlabel("near-surface RH (%)"); a.invert_xaxis(); a.legend(fontsize=8)
for rv, rh in zip(C.rv.values, C.RH.values):
    ax[0].annotate(f"{rv:g}", (rh, 100 * float(C.fraction.sel(rv=rv, group=groups[0]))), fontsize=7,
                   textcoords="offset points", xytext=(3, 3))
fig.suptitle(f"mask {MASK}{MASK_PARAMS}  (x axis: drier to the right; labels = r_v)")
fig.tight_layout(); plt.show()

## D. Local stability: Emanuel (1994) eqs. 6.1.7 and 6.2.10
Computed from the composite (mean) profiles of each group, with the run's mean pressure from its STAT file. Code: `samheat/analysis/stability.py`; tests: `tests/test_stability.py`.

* **N²** (6.1.7, unsaturated) $= g\,\partial \ln\theta_v/\partial z$, with the actual vapour $r_v$ = QV.
* **N_m²** (6.2.10, saturated) $= \frac{1}{1+r_T}\left[\Gamma_m \frac{\partial s^*}{\partial z} - (c_l\Gamma_m \ln T + g)\frac{\partial r_T}{\partial z}\right]$. This answers "how stable would this layer be **if saturated**?", with $r_T = r^*(T,p)$ + condensate. It splits exactly into
  * **entropy term** $\Gamma_m (\partial s^*/\partial z)_{r_T}/(1+r_T)$, the classic conditional-instability test (lapse rate vs the moist adiabat); and
  * **loading term** $-g\,(\partial r_T/\partial z)/(1+r_T)$, stabilising because in Emanuel's reversible framework a lifted parcel keeps all its water (about 2×10⁻⁵ s⁻² near the surface).
* Reading: N² > 0 and N_m² < 0 means **conditionally unstable**; N² < 0 means absolutely unstable.
* As in the book: liquid water only (no ice), reversible (no precipitation fallout). Constants are Emanuel's.

In [ ]:
# ---- D. Local stability (Emanuel 1994): N^2 (6.1.7) and N_m^2 (6.2.10) per group, one line per r_v ----
import numpy as np, matplotlib.pyplot as plt
from samheat.analysis.stability import stability_profiles
from samheat.analysis.composites import fit_xlim
CONDENSATE = ("QN",)          # condensate counted in r_T for N_m^2; ("QN", "QP") also counts falling rain
STAB_GROUPS = list(C.group.values)          # e.g. ["all", "updraft"]
PANELS = ["N2", "N2_m", "N2_m_entropy", "N2_m_loading"]

if not np.isfinite(C.p).all():
    print("WARNING: no mean pressure for some runs (STAT file missing?): stability will be NaN there")
ST = stability_profiles(C, CONDENSATE)

for g in STAB_GROUPS:
    fig, axs = plt.subplots(1, len(PANELS), figsize=(4 * len(PANELS), 4.5), sharey=True)
    for ax, q in zip(axs, PANELS):
        shown = []
        for i, rv in enumerate(ST.rv.values):
            prof = 1e4 * ST[q].sel(rv=rv, group=g)
            ax.plot(prof, ST.z / 1000, c=RV_COLORS[i % len(RV_COLORS)], label=f"rv {rv:g} (RH {float(C.RH.sel(rv=rv)):.0f} %)")
            shown.append(prof.values)
        fit_xlim(ax, shown + [np.zeros(ST.sizes["z"])], ST.z, ZMAX_PLOT)
        ax.axvline(0, c="k", lw=.6)
        ax.set_title(q); ax.set_xlabel("1e-4 s$^{-2}$")
    axs[0].set_ylabel("z (km)")
    if ZMAX_PLOT:
        axs[0].set_ylim(0, ZMAX_PLOT / 1000)
    axs[0].legend(fontsize=7)
    fig.suptitle(f"{g}: stability (Emanuel 6.1.7 / 6.2.10), r_T = r* + {'+'.join(CONDENSATE)}; mask {MASK}{MASK_PARAMS}")
    fig.tight_layout(); plt.show()

## Appendix: original testing cells (from Untitled-8)
Kept as they were. They use `data` from their own cells.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

In [ ]:
file_test = '/scratch/ttg2015/Torch/ExtremeHeat/SCM/SAM_HEAT_runs/vdd2025_vddgrid/stage2_rv0/OUT_3D/FWRCE_stage2_rv0_32_*.nc'

In [ ]:
file_test_open = xr.open_mfdataset(file_test)

In [ ]:
data = file_test_open
data

In [ ]:
MSE = 1.005 * data.TABS + 2.5 * data.QV + (9.8/1000) * data.z

In [ ]:
MSE.mean(['x','y','time']).plot(y = 'z')
plt.ylim(0,10000)
plt.xlim(300,350)

In [ ]:
fig, ax = plt.subplots(ncols = 3, figsize = (18,5))
data.W.isel(z = 1, time = 0).plot(ax = ax[0])
data.W.isel(z = 2, time = 0).plot(ax = ax[1])
data.W.isel(z = 3, time = 0).plot(ax = ax[2])

In [ ]:
print(data.time.shape)

In [ ]:
# ---- Make an mp4 of W at z = 1, 2, 3 (same panels as above) ----
import matplotlib.pyplot as plt
from matplotlib import animation
LEVELS = [1, 2, 3]
OUT = "W_levels_1-3.mp4"
FPS = 10

W = data.W.isel(z=LEVELS).load()              # read these 3 levels for all times once
VMAX = float(abs(W).max())                     # one fixed colour scale for every frame (else the colours jump)

fig, ax = plt.subplots(ncols=3, figsize=(18, 5))
meshes = [W.isel(z=i, time=0).plot(ax=ax[i], vmin=-VMAX, vmax=VMAX, cmap="RdBu_r") for i in range(3)]

def frame(t):
    for i, m in enumerate(meshes):
        snap = W.isel(z=i, time=t)
        m.set_array(snap.values)
        ax[i].set_title(f"z = {float(snap.z):.1f} m, time = {float(snap.time):.3f} d")
    return meshes

anim = animation.FuncAnimation(fig, frame, frames=W.sizes["time"], blit=False)

if not animation.writers.is_available("ffmpeg"):
    try:
        import imageio_ffmpeg                      # pip install imageio-ffmpeg  (bundles an ffmpeg binary)
        plt.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
    except ImportError:
        pass
if animation.writers.is_available("ffmpeg"):
    anim.save(OUT, writer=animation.FFMpegWriter(fps=FPS), dpi=100)
else:
    OUT = OUT.replace(".mp4", ".gif")              # no ffmpeg anywhere: a GIF still works
    anim.save(OUT, writer=animation.PillowWriter(fps=FPS), dpi=80)
plt.close(fig)
print("saved", OUT)

In [ ]:
data.W.isel(z = 1).plot.hist(bins =1000);
plt.axvline( 0.3, c = 'k')
plt.axvline( -0.3, c = 'k')

In [ ]:
import numpy as np, pandas as pd

rows = []
for k in range(data.sizes["z"]):
    w = data.W.isel(z=k).values.astype("float64").ravel()   # every x, y, time at this level
    m, s = w.mean(), w.std()                                  # same as your .mean() / .std() (ddof=0)
    skew = ((w - m) ** 3).mean() / s**3 if s > 0 else np.nan
    rows.append(dict(k=k, z_m=float(data.z[k]), mean=m, std=s, skew=skew))

stats = pd.DataFrame(rows).set_index("k")
print(stats.round(4).to_string())

In [ ]:
# ---- Composite vertical profiles of updrafts / downdrafts / intermediate, classified by W at one level ----
# Expects `data` as opened by samheat.io.sam3d.open_run_3d: 3D fields (time, z, y, x) and p (time, z) in hPa.
import numpy as np, xarray as xr

K_MASK = 1        # level used to classify (W at isel z=1)
X = 0.3           # m/s: updraft W > X, downdraft W < -X, intermediate |W| <= X

# SAM constants (params.f90) and its liquid/ice partition (MICRO_SAM1MOM/micro_params.f90)
CP, G, LC, LF = 1004., 9.81, 2.5104e6, 0.3336e6
TBGMIN, TBGMAX, TPRMIN, TPRMAX = 253.16, 273.16, 268.16, 283.16

FIELDS = [v for v in data.data_vars if data[v].dims == ("time", "z", "y", "x")]   # U V W PP QRAD TABS QV QN QP
z = data.z.values.astype(float)
zz = z[:, None, None]

def derived(f, p):
    """Extra fields for one snapshot. f: dict of 3D arrays (z, y, x) in SAM units; p: (z,) hPa."""
    T, qv, qn, qp = f["TABS"], f["QV"], f["QN"], f["QP"]                 # K, g/kg, g/kg, g/kg
    out = {}
    # 1. simple MSE (kJ/kg), exactly as defined
    out["MSE"] = 1.005 * T + 2.5 * qv + (9.8 / 1000) * zz
    # 2. frozen MSE (kJ/kg): cp T + g z + Lc qv - Lf q_ice; conserved under condensation, evaporation
    #    and freezing. Ice fractions follow SAM's temperature partition of QN and QP.
    ice_n = 1 - np.clip((T - TBGMIN) / (TBGMAX - TBGMIN), 0, 1)
    ice_p = 1 - np.clip((T - TPRMIN) / (TPRMAX - TPRMIN), 0, 1)
    q_ice = (ice_n * qn + ice_p * qp) / 1000.                            # kg/kg
    q_liq = ((1 - ice_n) * qn + (1 - ice_p) * qp) / 1000.               # kg/kg
    out["MSE_frozen"] = (CP * T + G * zz + LC * qv / 1000. - LF * q_ice) / 1000.
    # 3. SAM's own conserved energy variable, liquid/ice static energy (kJ/kg)
    out["S_LI"] = (CP * T + G * zz - LC * q_liq - (LC + LF) * q_ice) / 1000.
    # 4. equivalent potential temperature (K), Bolton (1980) eqs. 22 and 39, with p + PP
    ptot = p[:, None, None] + f["PP"] / 100.                             # hPa
    e = ptot * (qv / 1000.) / (0.622 + qv / 1000.)                       # vapour pressure, hPa
    TL = 2840. / (3.5 * np.log(T) - np.log(np.maximum(e, 1e-6)) - 4.805) + 55.
    out["THETA_E"] = (T * (1000. / ptot) ** (0.2854 * (1 - 0.28e-3 * qv))
                      * np.exp((3.376 / TL - 0.00254) * qv * (1 + 0.81e-3 * qv)))
    return out

groups = ("updraft", "downdraft", "intermediate")
sums, count = {}, {g: 0 for g in groups}
nt = data.sizes["time"]
for t in range(nt):
    s = data.isel(time=t).load()                                         # one snapshot (~40 MB)
    f = {v: s[v].values.astype(np.float64) for v in FIELDS}             # each (z, y, x)
    p = s["p"].values.astype(np.float64)                                 # (z,)
    f |= derived(f, p)
    w0 = f["W"][K_MASK]                                                  # (y, x)
    masks = dict(updraft=w0 > X, downdraft=w0 < -X, intermediate=np.abs(w0) <= X)
    for g, m in masks.items():
        count[g] += int(m.sum())
        for v, a in f.items():
            sums[(g, v)] = sums.get((g, v), 0.) + a[:, m].sum(axis=1)  # (z,)
    if t % 25 == 0 or t == nt - 1:
        print(f"snapshot {t + 1}/{nt}")

total = sum(count.values())
names = list(dict.fromkeys(v for _, v in sums))
profiles = xr.Dataset(
    {v: (("group", "z"), np.array([sums[(g, v)] / max(count[g], 1) for g in groups])) for v in names},
    coords=dict(group=list(groups), z=z, fraction=("group", [count[g] / total for g in groups])))
print(f"\nclassified by W at z index {K_MASK} (labelled z = {z[K_MASK]:.1f} m), X = {X} m/s, "
      f"days {float(data.time[0]):.2f}-{float(data.time[-1]):.2f}")
for g in groups:
    print(f"  {g:12s} {100 * count[g] / total:6.2f} % of columns")
profiles

In [ ]:
# ---- Plot the composite profiles: one panel per quantity, one line per group ----
import matplotlib.pyplot as plt
qs = list(profiles.data_vars)
ncol = 4
nrow = -(-len(qs) // ncol)
fig, axs = plt.subplots(nrow, ncol, figsize=(4 * ncol, 4.5 * nrow), sharey=True, squeeze=False)
for ax, q in zip(axs.flat, qs):
    for g in profiles.group.values:
        ax.plot(profiles[q].sel(group=g), profiles.z / 1000, label=f"{g} ({100 * float(profiles.fraction.sel(group=g)):.1f} %)")
    ax.set_title(q)
for ax in axs[:, 0]:
    ax.set_ylabel("z (km)")
for ax in axs.flat[len(qs):]:
    ax.set_visible(False)
axs.flat[0].legend(fontsize=8)
fig.suptitle(f"Composites by W at z index {K_MASK}, |W| > {X} m/s")
fig.tight_layout()
plt.show()

In [ ]:
profiles.MSE.sel(group = 'updraft').plot(y = 'z', color = 'b', label = 'updraft')
profiles.MSE.sel(group = 'downdraft').plot(y = 'z', color = 'r', label = 'downdraft')
profiles.MSE.sel(group = 'intermediate').plot(y = 'z', c = 'g', label = 'intermediate')

plt.ylim(0,10000)
plt.xlim(320,340)
plt.legend()
plt.show()

profiles.MSE_frozen.sel(group = 'updraft').plot(y = 'z', color = 'b', label = 'updraft')
profiles.MSE_frozen.sel(group = 'downdraft').plot(y = 'z', color = 'r', label = 'downdraft')
profiles.MSE_frozen.sel(group = 'intermediate').plot(y = 'z', c = 'g', label = 'intermediate')

plt.ylim(0,10000)
plt.xlim(320,340)
plt.legend()
plt.show()
profiles.THETA_E.sel(group = 'updraft').plot(y = 'z', color = 'b', label = 'updraft')
profiles.THETA_E.sel(group = 'downdraft').plot(y = 'z', color = 'r', label = 'downdraft')
profiles.THETA_E.sel(group = 'intermediate').plot(y = 'z', c = 'g', label = 'intermediate')

plt.ylim(0,10000)
plt.xlim(320,340)
plt.legend()

In [ ]:
profiles